# Solar panel defect detector — RGB only

### What this model can and cannot do

Trained on visible-light imagery, so it detects what visible light shows:

**Can** — soiling and dust, bird droppings, cracked or shattered glass, delamination and
discoloration, vegetation shading, snow cover, missing or displaced modules.

**Cannot** — hotspots, cell and multi-cell defects, bypass-diode failure, offline modules.
These are *electrical* faults. They are visible in thermal infrared and essentially
invisible in RGB. No amount of training data fixes that; it is physics, not modelling. The
web app states this limitation to users, and so should you if you show this to anyone.

If you later add a thermal camera, the strongest public dataset is
[RaptorMaps InfraredSolarModules](https://github.com/RaptorMaps/InfraredSolarModules) —
20,000 real IR images, 12 classes, already cropped to single modules. Those crops suit a
classifier rather than a detector, which is a different (and much cheaper) pipeline.

### Getting data

You have no imagery of your own, so this trains on public sets. Roboflow Universe has
several RGB solar panel detection and soiling datasets; a free account gives you an API key
and export in YOLO format.

**Audit whatever you download before training on it.** The turbine dataset scored 0.782 and
was useless because of defects `tools/audit_dataset.py` catches in seconds. Assume a
scraped dataset has the same problems until the audit says otherwise.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
import os, re, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/abyyworld/Drone-visualisation-training.git"
BRANCH   = "claude/model-retrain-solar-panels-n77z3v"
WORK     = Path("/kaggle/working/drone-inspection")

# Same self-cloning setup as the turbine notebook: no Kaggle Dataset needs attaching,
# and the tools are always the current ones rather than whatever was attached last.
def looks_like_repo(p: Path) -> bool:
    return (p / "tools" / "audit_dataset.py").exists()

source = None
for entry in sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []:
    if entry.is_dir():
        for candidate in [entry] + [d for d in entry.iterdir() if d.is_dir()]:
            if looks_like_repo(candidate):
                source = candidate
                break
    if source:
        break

if source:
    print(f"Found the repo at {source}")
    if not WORK.exists():
        shutil.copytree(source, WORK)
else:
    print("No attached input looks like the repo - cloning from GitHub.")
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(WORK)], check=True)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
assert looks_like_repo(WORK), f"Repo incomplete at {WORK}"
print(f"OK -> {WORK}")

## 1. Get the data

Roboflow Universe hosts several RGB solar-defect sets. The cell below downloads one on Kaggle (which has internet), so nothing touches your laptop.

**Look at the images before you train on them.** The turbine v1 model scored mAP50 0.782 and was useless, because nobody checked what it had actually learned. The audit in the next section catches the structural failures; only your eyes catch "these are stock photos, not drone imagery".

In [ ]:
# ---------------------------------------------------------------------------------------
# Download the solar dataset. Runs on Kaggle, which has internet; nothing needs downloading
# to your laptop and no Kaggle Dataset needs attaching.
#
# ONE-TIME SETUP (2 minutes):
#   1. Sign in at https://universe.roboflow.com (free, Google sign-in works).
#   2. Open the dataset you want, click "Download Dataset", format "YOLOv11".
#      It shows a snippet containing workspace(...), project(...) and version(...).
#   3. Copy your API key from https://app.roboflow.com/settings/api
#   4. In this notebook: Add-ons > Secrets > add ROBOFLOW_API_KEY. Never paste it inline -
#      a committed notebook is public and a leaked key is a rotated key.
#
# CANDIDATES I FOUND BUT COULD NOT VERIFY (the container this was written in cannot reach
# Roboflow, so treat these as leads, not recommendations - open them and look at the images
# before committing hours of GPU time):
#   https://universe.roboflow.com/solarpanel-2me5p/solar-panel-defects-lnge0
#   https://universe.roboflow.com/solar-panel-a5cyn/dusty-and-bird-droppings
#   https://universe.roboflow.com/solar-panel-damage-classification-crto7/solar-panels-damage
#   https://universe.roboflow.com/search?q=class%3Asolar+panel
#
# The published comparison (Solar 2025, 5(1), 6) reports mAP50 0.934 with YOLOv11 on a
# 6,493-image RGB set with classes bird_drop / cracked / dusty / panel. If you find that
# exact set, prefer it: a published baseline means a bad result is your pipeline, not a
# mystery.
# ---------------------------------------------------------------------------------------

# Paste EITHER the Universe URL, OR fill in the triple from the download snippet.
DATASET_URL = ""          # e.g. "https://universe.roboflow.com/solarpanel-2me5p/solar-panel-defects-lnge0"
WORKSPACE   = ""          # overrides the URL if set
PROJECT     = ""
VERSION     = None        # int; None = highest available

SOLAR = Path("/kaggle/working/solar_raw")

if not SOLAR.exists():
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow

    workspace, project_id, version = WORKSPACE, PROJECT, VERSION
    if not (workspace and project_id):
        match = re.search(r"universe\.roboflow\.com/([^/]+)/([^/?#]+)", DATASET_URL)
        assert match, (
            "Set DATASET_URL to a Roboflow Universe dataset URL, or fill in "
            "WORKSPACE/PROJECT from the download snippet. See the notes above."
        )
        workspace, project_id = match.group(1), match.group(2)
        tail = re.search(r"/dataset/(\d+)", DATASET_URL)
        if tail and version is None:
            version = int(tail.group(1))

    key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    project = Roboflow(api_key=key).workspace(workspace).project(project_id)

    if version is None:
        # Version numbering is not always 1..n, so ask rather than assume.
        numbers = []
        for v in project.versions():
            raw = getattr(v, "version", None) or getattr(v, "id", "")
            digits = re.search(r"(\d+)$", str(raw))
            if digits:
                numbers.append(int(digits.group(1)))
        assert numbers, "Could not list versions - set VERSION explicitly from the snippet."
        version = max(numbers)
        print(f"Using version {version} (available: {sorted(numbers)})")

    project.version(version).download("yolov11", location=str(SOLAR))

assert (SOLAR / "data.yaml").exists(), f"No data.yaml under {SOLAR} - download did not land."

import yaml
spec = yaml.safe_load((SOLAR / "data.yaml").read_text())
print(f"\nclasses ({spec['nc']}): {spec['names']}")
for split in ("train", "valid", "test"):
    d = SOLAR / split / "images"
    print(f"  {split:<6}{len(list(d.iterdir())) if d.is_dir() else 0:>6} images")

## Audit before training

Run this on the downloaded data. Two failures matter most:

* **source-family shortcut** — if the filenames reveal the class, or a class never shares an
  image with another, the model will learn the shortcut instead of the defect. This is
  exactly what ruined the turbine v1 model.
* **near-duplicate frame leakage** — solar farm imagery is usually captured as a flight
  sequence, so a random split puts adjacent frames on both sides. Validation then measures
  memorisation.

If either fails, re-split by capture group before training. `tools/rebuild_turbine.py` shows
the pattern; the contiguous-block split in `block_split()` is directly reusable.

In [ ]:
!python3 tools/audit_dataset.py {SOLAR}

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(SOLAR / "data.yaml"),
    epochs=100,
    imgsz=960,             # soiling patches and cell cracks are small
    batch=8,
    # patience MUST outlast warmup by a wide margin. The turbine run used
    # warmup=5/patience=10, peaked at epoch 2 while the LR was still ramping,
    # and was killed at epoch 12 having never trained at full LR - mAP50 0.194.
    patience=30,

    optimizer="SGD",
    lr0=0.005, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3, cos_lr=True,

    cache=False, device=0,   # not "disk": it deadlocked the turbine run at epoch 47
     workers=2, seed=0, deterministic=True,

    # Panels are rectilinear and photographed from many angles; rotation helps here more
    # than it does on blades. Keep hue jitter low - discoloration IS the signal for one of
    # the classes, so distorting colour trains the model to ignore what it should detect.
    degrees=15.0, fliplr=0.5, flipud=0.5, scale=0.4,
    hsv_h=0.005, hsv_s=0.5, hsv_v=0.3,
    mosaic=1.0, close_mosaic=10,

    save_period=10,
    project="/kaggle/working/runs", name="solar_v1", exist_ok=True,
    plots=True,
)

In [ ]:
import subprocess
# Not `!cmd \` across lines: Kaggle's IPython transform truncates the command at the
# backslash and leaves the following lines as invalid Python.
subprocess.run([
    "python3", "tools/evaluate.py",
    "/kaggle/working/runs/solar_v1/weights/best.pt",
    "--data", str(SOLAR / "data.yaml"), "--split", "test", "--imgsz", "960",
    "--out", "/kaggle/working/runs/eval_solar",
], check=True)


## Export

`export_onnx.py` rewrites `web/models/manifest.json` with the class list from the trained
model, so the browser labels match the weights. Afterwards, **set the severity weights by
hand** — the exporter defaults every class to 1.0, and engineering judgement is what makes
that number mean something. A missing module is not the same event as some dust.

In [ ]:
import subprocess
# Not `!cmd \` across lines: Kaggle's IPython transform truncates the command at the
# backslash and leaves the following lines as invalid Python.
subprocess.run([
    "python3", "tools/export_onnx.py",
    "/kaggle/working/runs/solar_v1/weights/best.pt",
    "--name", "solar", "--imgsz", "960",
], check=True)

subprocess.run(["ls", "-lh", "web/models/"], check=True)
